# 08 - Fault category: TF-IDF + logistic regression

A fast alternative to the MiniLM classifier (06/07). Same data, same 10 categories in the same order.
TF-IDF word 1-2 grams + balanced logistic regression, temperature-calibrated, exported to ONNX with skl2onnx.
Evaluation: 5-fold stratified CV by sentence, 3 shuffle seeds, threshold 0.80 on the top probability.

In [ ]:
import json, time
from pathlib import Path
import numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from ml_research.report_category_data import prepare, CATEGORIES

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data/report-classifier/municipal_training_set_1100.csv"
OUT = ROOT / "artifacts/fault-category-tfidf"; OUT.mkdir(parents=True, exist_ok=True)
THRESHOLD, SEEDS, K = 0.80, [0, 1, 2], 5
# ASCII-only token pattern: sklearn (Python re) and onnxruntime (re2) agree on it; \w differs on accented letters.
TOKEN_PATTERN = r"[a-z0-9_]{2,}"
sentences = prepare(DATA)
X, y = sentences["sentence"].to_numpy(), sentences["target"].to_numpy()
print(len(sentences)); print(sentences["label"].value_counts().sort_index().to_string())

In [ ]:
def word_pipe(C=10.0):
    return Pipeline([("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=True,
                                               sublinear_tf=True, token_pattern=TOKEN_PATTERN)),
                     ("lr", LogisticRegression(C=C, class_weight="balanced", max_iter=5000))])
def char_pipe(C=10.0):
    return Pipeline([("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=True, sublinear_tf=True)),
                     ("lr", LogisticRegression(C=C, class_weight="balanced", max_iter=5000))])

def oof_logits(make, seed):
    logits = np.zeros((len(X), len(CATEGORIES)))
    for tr, te in StratifiedKFold(K, shuffle=True, random_state=seed).split(X, y):
        m = make().fit(X[tr], y[tr]); logits[te] = m.decision_function(X[te])
    return logits

def fit_temperature(logits, labels):
    nll = lambda T: -log_softmax(logits / T, axis=1)[np.arange(len(labels)), labels].mean()
    return minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x

def ece(p, labels, bins=10):
    conf, pred = p.max(1), p.argmax(1); e = 0.0
    for lo in np.linspace(0, 1, bins + 1)[:-1]:
        s = (conf > lo) & (conf <= lo + 1 / bins)
        if s.any(): e += s.mean() * abs((pred[s] == labels[s]).mean() - conf[s].mean())
    return e

def report(p, labels):
    conf, pred = p.max(1), p.argmax(1); kept = conf >= THRESHOLD
    rows = {}
    for i, c in enumerate(CATEGORIES):
        tp = ((pred == i) & kept & (labels == i)).sum(); fp = ((pred == i) & kept & (labels != i)).sum()
        fn = (labels == i).sum() - tp
        P = tp / (tp + fp) if tp + fp else float("nan"); R = tp / (tp + fn)
        rows[c] = dict(precision=P, recall=R, f1=0.0 if not (P + R) or np.isnan(P) else 2 * P * R / (P + R))
    df = pd.DataFrame(rows).T
    return df, dict(macro_f1=df["f1"].mean(), coverage=kept.mean(), accuracy_all=(pred == labels).mean(),
                    accuracy_kept=(pred[kept] == labels[kept]).mean(), ece=ece(p, labels))

In [ ]:
def evaluate(make, name):
    tables, summ, temps = [], [], []
    for seed in SEEDS:
        lg = oof_logits(make, seed)
        # Temperature fitted on this seed's out-of-fold logits (one scalar; mild optimism noted in README).
        T = fit_temperature(lg, y); temps.append(T)
        raw_ece = ece(softmax(lg, 1), y)
        t, s = report(softmax(lg / T, 1), y); s["ece_uncalibrated"] = raw_ece
        tables.append(t); summ.append(s)
    table = sum(tables) / len(tables); s = pd.DataFrame(summ).mean().to_dict(); s["temperature"] = float(np.mean(temps))
    print(name, {k: round(v, 4) for k, v in s.items()}); return table, s

results = {}
for C in [1.0, 10.0, 30.0]:
    results[f"word_C{C}"] = evaluate(lambda C=C: word_pipe(C), f"word 1-2 C={C}")
for C in [10.0]:
    results[f"char_C{C}"] = evaluate(lambda C=C: char_pipe(C), f"char_wb 3-5 C={C}")

In [ ]:
best = max((k for k in results if k.startswith("word")), key=lambda k: results[k][1]["macro_f1"])
table, summary = results[best]; print("chosen:", best); print(table.round(3)); print(summary)
print("miss 0.95 precision:", [c for c in CATEGORIES if c != "OTHER" and not table.loc[c, "precision"] >= 0.95])

In [ ]:
C = float(best.split("_C")[1]); T = summary["temperature"]
final = word_pipe(C).fit(X, y)
# Fold the temperature into the linear layer so the exported model emits calibrated probabilities directly.
final.named_steps["lr"].coef_ /= T; final.named_steps["lr"].intercept_ /= T

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType
onx = convert_sklearn(final, initial_types=[("text", StringTensorType([None, 1]))],
                      options={id(final.named_steps["lr"]): {"zipmap": False}}, target_opset=17)
names = [o.name for o in onx.graph.output]; print("outputs before rename:", names)
prob = names[1]
for node in onx.graph.node:
    node.output[:] = ["probabilities" if o == prob else o for o in node.output]
for o in onx.graph.output:
    if o.name == prob: o.name = "probabilities"
onx.graph.output.remove([o for o in onx.graph.output if o.name != "probabilities"][0])
(OUT / "model.onnx").write_bytes(onx.SerializeToString()); size = (OUT / "model.onnx").stat().st_size; print(size)

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(str(OUT / "model.onnx"))
print([(i.name, i.shape, i.type) for i in sess.get_inputs()], [(o.name, o.shape, o.type) for o in sess.get_outputs()])
def onnx_probs(texts): return sess.run(["probabilities"], {"text": np.array(texts, dtype=object).reshape(-1, 1)})[0]
odd = ["", "   ", "!!!???...", "漏水了 水管爆了 💧💧", "🚨🚨🚨", "Lift stuck between floors 5 and 6", "The water pump is very noisy at night",
       "Gate won't close at the car park", "LIGHT!!! broken!!!", "water-leak, ceiling; dripping.", "x" * 5000,
       " ".join(["pothole on the road"] * 300), "Café naïve résumé", "1234 5678"]
checks = list(X) + odd
diffs = np.abs(final.predict_proba(np.array(checks, dtype=object)) - onnx_probs(checks)).max(1)
print("max diff all:", diffs.max(), "sentences:", diffs[:len(X)].max(), "odd:", diffs[len(X):].max())
for t, d in zip(odd, diffs[len(X):]): print(repr(t[:40]), d)

In [ ]:
per_cat = [sentences[sentences["label"] == c]["sentence"].iloc[0] for c in CATEGORIES]
estate = ["Lift stuck between floors 5 and 6", "The water pump is very noisy at night", "Gate won't close at the car park",
          "Corridor light on level 3 keeps flickering", "Rubbish chute is blocked and smelly", "Someone sprayed paint on the void deck wall"]
fixture_texts = per_cat + estate + odd[:5] + odd[8:] + ["Car parked across two lots blocking the lane", "Drain is clogged and flooding the path"]
P = onnx_probs(fixture_texts)
fixture = dict(model="fault-category-tfidf-v1", categories=CATEGORIES, tolerance=1e-5,
               cases=[dict(text=t, probabilities=[float(v) for v in p]) for t, p in zip(fixture_texts, P)])
(OUT / "parity-fixture.json").write_text(json.dumps(fixture, ensure_ascii=False, indent=2)); print(len(fixture_texts))
for t, p in zip(fixture_texts, P): print(f"{CATEGORIES[p.argmax()]:16s} {p.max():.2f} {t[:50]!r}")

In [ ]:
metrics = dict(version="fault-category-tfidf-v1", threshold=THRESHOLD, folds=K, seeds=SEEDS, n_sentences=int(len(X)), chosen=best,
               settings=dict(analyzer="word", ngram_range=[1, 2], lowercase=True, sublinear_tf=True, token_pattern=TOKEN_PATTERN,
                             C=C, class_weight="balanced", temperature=T),
               cv_summary={k: {kk: float(vv) for kk, vv in s.items()} for k, (_, s) in results.items()},
               per_class={c: {k: float(v) for k, v in r.items()} for c, r in table.to_dict("index").items()},
               onnx=dict(max_abs_diff=float(diffs.max()), size_bytes=size, input="text string [N,1]", output="probabilities float [N,10]"))
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2)); print("done")